In [0]:

# Healthcare Data Engineering - Bronze Layer
# Step 1: Verify Spark environment

# print("Healthcare Bronze Ingestion Started")

# print("Spark Version:", spark.version)

# test_data = [
#     (1, "Patient_A", "FL"),
#     (2, "Patient_B", "TX"),
#     (3, "Patient_C", "CA")
# ]

# df_test = spark.createDataFrame(
#     test_data,
#     ["patient_id", "patient_name", "state"]
# )

# display(df_test)


In [0]:

# # Step 2: Check ADLS Gen2 access

# storage_account = "sthealthcarede2026"
# container = "bronze"

# file_path = (
#     f"abfss://{container}@{storage_account}"
#     ".dfs.core.windows.net/patients/patients.csv"
# )

# print("Checking file:", file_path)

# try:
#     files = dbutils.fs.ls(
#         f"abfss://{container}@{storage_account}.dfs.core.windows.net/patients/"
#     )

#     for file in files:
#         print(file.name, file.size)

# except Exception as e:
#     print("Storage access check failed.")
#     print(str(e)[:1500])


In [0]:
#Create a source parameter widgets
dbutils.widgets.text("source_folder", "patients")
dbutils.widgets.text("source_container", "bronze")
dbutils.widgets.text("file_name", "patients.csv")

# Create a target parameter widgets
dbutils.widgets.text("target_schema", "bronze")
dbutils.widgets.text("target_table", "patients_raw")

#Reading a source parameter
source_container = dbutils.widgets.get("source_container")
source_folder = dbutils.widgets.get("source_folder")
source_file = dbutils.widgets.get("file_name")

#Reading a target parameter
target_schema = dbutils.widgets.get("target_schema")
target_table = dbutils.widgets.get("target_table")

print("Source container: ", source_container)
print("Source folder:", source_folder)
print("File name: ", source_file)

In [0]:
# source file partial path

path = f"{source_folder}/{source_file}"

print(path)

# target table path

target_path = f"{target_schema}.{target_table}"
print(target_path)


In [0]:
dbutils.widgets.text("storage_account", "sthealthcarede2026")
storage_account = dbutils.widgets.get("storage_account")
print("Storage account: ", storage_account)

In [0]:
#abfss://bronze@sthealthcarede2026.dfs.core.windows.net/patients/patients.csv
bronze_path = f"abfss://{source_container}@{storage_account}.dfs.core.windows.net/{path}"
print(bronze_path)

In [0]:
df_bronze_raw = (
    spark.read
    .option("header", "true")
    .option("inferschema", "true")
    .csv(bronze_path)
)

df_bronze_raw.show(10, truncate = False)
print("Total records :", df_bronze_raw.count())   # this is to get the total number of records
print("Total columns :" , len(df_bronze_raw.columns))    # this is to get the total number of columns

In [0]:
from pyspark.sql.functions import lit, current_timestamp
dbutils.widgets.text("batch_id", "BATCH_20261102_001")
batch_id = dbutils.widgets.get("batch_id")

df_bronze = (
    df_bronze_raw
    .withColumn("batch_id", lit(batch_id))
    .withColumn("source_file", lit(bronze_path))
    .withColumn("ingestion_timestamp", current_timestamp())
)

df_bronze.show(5, truncate = False)
print("Total number of columns :", len(df_bronze.columns))

In [0]:
bronze_count =  df_bronze.count() 
print("bronze record count: ", bronze_count)
print("bronze dataframe column count: ", len(df_bronze.columns))

required_columns = {"patient_id", "batch_id", "source_file"}

actual_columns = set(df_bronze.columns)
print("Actual Columns: ", actual_columns)


# missing columns
# Business rule: If missing_columns is not empty, raise a ValueError.
missing_columns = required_columns - actual_columns
print("Missing columns: ", missing_columns)


if missing_columns:
    raise ValueError(f"Missing required columns : {sorted(missing_columns)}")

if bronze_count == 0:
    raise ValueError("No records found in bronze dataframe")


from pyspark.sql.functions import col
df_patientID_null = (
    df_bronze
    .filter(col("patient_id").isNull())
)
null_patientids_count = df_patientID_null.count()
print("Null PatientIDs count: ", null_patientids_count)

if null_patientids_count >0:
    raise ValueError("Null patientID's are present in bronze dataframe")

df_duplicate_patients = (
    df_bronze
    .groupBy(col("patient_id"))
    .count()
    .filter(col('count') > 1)
)
duplicate_patients_count = df_duplicate_patients.count()
print("duplicate patients count: ", duplicate_patients_count)

if duplicate_patients_count > 0:
    raise ValueError("duplicate patientID's are present in bronze dataframe")

print("Bronze validation passed")

In [0]:
catalog_name = spark.sql(
    "select current_catalog()"
).first()[0]

print("Catalog: ", catalog_name)

full_table_name = f"{catalog_name}.{target_path}"
print(full_table_name)


In [0]:
if not spark.catalog.tableExists(full_table_name):
    (
        df_bronze
        .write
        .format("delta")
        .mode("append")
        .saveAsTable(full_table_name)
    )

    print(" Bronze table Created successfully! ")

else:
    print(" Bronze table already exists")

In [0]:
print("Table exists:", spark.catalog.tableExists(full_table_name))

df_verify = spark.table(full_table_name)

print("Total records:", df_verify.count())
print("Total columns:", len(df_verify.columns))

df_verify.show(5, truncate=False)




In [0]:
%sql
DESCRIBE HISTORY dbw_healthcare_de_dev.bronze.patients_raw;

In [0]:
df_history = spark.sql(f"DESCRIBE HISTORY {full_table_name}")
display(df_history)